In [0]:
def processInvoiceAddendumDetails(df_CycleData_AGG,UpdatedBy="ADB_UserSubscription"):
    
    df_InvoiceAddendumDetails = (spark.read.table('Promotion.FACT_InvoiceAddendumDetails')
                                            .filter('VersionId=1')
                                            .groupBy('ConsumerSubscriptionUUID')
                                            .sum('IncrementalInvoiceCharge')
                                            .withColumnRenamed('sum(IncrementalInvoiceCharge)','SubscriptionInvoiceAmt'))
    df_PatientClassification = (spark.read.table('Promotion.DIM_PatientClassification')
                                .select('PatientClassificationName','PromotionUUID', 'PatientClassificationUUID', 'ListPrice', 'InvoiceCalculationType', 
                        col('EffectiveDate').alias('PCL_EffectiveDate'),
                        col('TerminationDate').alias('PCL_TerminationDate'))
                                )
    df_ConsumerSubScription = spark.read.table('Promotion.FACT_ConsumerSubscription').filter('VersionId=1')
   
    df_InvoiceAddendum_Sub_Agg = (df_ConsumerSubScription
                                .join(df_InvoiceAddendumDetails,'ConsumerSubscriptionUUID','left')
                                .select(df_ConsumerSubScription.CoolSculptingID,
                                        df_ConsumerSubScription.ShipToAccountUUID,
                                        df_ConsumerSubScription.SoldToAccountID,
                                        df_ConsumerSubScription.PromotionUUID,
                                        df_ConsumerSubScription.SubscriptionStartDate,
                                        df_ConsumerSubScription.SubscriptionEndDate,
                                        df_ConsumerSubScription.ConsumerSubscriptionUUID,
                                        df_ConsumerSubScription.PilotSubscriptionFlag,
                                        df_ConsumerSubScription.Comments.alias('ConsumerSubscription_Comments'),
                                        df_InvoiceAddendumDetails.SubscriptionInvoiceAmt
                                        )
                                )

    df_CycleData_Subscription = (df_CycleData_AGG
                                #  .filter('NoPhoneNumberFlag = 0')
                        .join(df_InvoiceAddendum_Sub_Agg,
                                            (df_CycleData_AGG.CoolSculptingID == df_InvoiceAddendum_Sub_Agg.CoolSculptingID) &
                                            (df_CycleData_AGG.ShipToAccountUUID == df_InvoiceAddendum_Sub_Agg.ShipToAccountUUID) &
                                            (df_CycleData_AGG.PromotionUUID == df_InvoiceAddendum_Sub_Agg.PromotionUUID) &
                                            (df_CycleData_AGG.CycleDate.between(df_InvoiceAddendum_Sub_Agg.SubscriptionStartDate,df_InvoiceAddendum_Sub_Agg.SubscriptionEndDate)) &
                                            ((df_CycleData_AGG.PerCycleFlag == 0) | (df_CycleData_AGG.PerCycleWhenNoSubscriptionFlag == 1)),
                                            'left'
                                            )
                        .select(df_CycleData_AGG.CoolSculptingID,
                                df_CycleData_AGG.ShipToAccountID,
                                df_CycleData_AGG.ShipToAccountUUID,
                                df_CycleData_AGG.SoldToAccountID,
                                df_CycleData_AGG.PromotionUUID,
                                df_CycleData_AGG.CycleDate,
                                df_CycleData_AGG.CycleCount,
                                df_CycleData_AGG.NoPhoneNumberFlag,
                                df_CycleData_AGG.MaxInvoiceDate,
                                df_CycleData_AGG.MinCycleTmstmp,
                                df_CycleData_AGG.FraudFlag,
                                df_CycleData_AGG.MapFlag,
                                df_CycleData_AGG.TNCFlag,
                                df_CycleData_AGG.TNCPerCycleFlag,
                                df_CycleData_AGG.PerCycleFeeFlag,
                                df_CycleData_AGG.PerCycleFlag,
                                df_CycleData_AGG.OffBoarding_NonP3Flag,
                                df_CycleData_AGG.OffBoarding_NonP3_TreatmentFlag,
                                df_CycleData_AGG.OffBoarding_NonP3PerCycleFlag,
                                df_CycleData_AGG.OffBoarding_NonP3_TreatmentPerCycleFlag,
                                df_CycleData_AGG.EfficacyFlag,
                                df_CycleData_AGG.SmartCardSerialNumber,
                                df_CycleData_AGG.ApplicatorSerialNumber,
                                df_CycleData_AGG.CycleID,
                                df_InvoiceAddendum_Sub_Agg.ConsumerSubscription_Comments,
                                df_InvoiceAddendum_Sub_Agg.ConsumerSubscriptionUUID,
                                df_InvoiceAddendum_Sub_Agg.SubscriptionInvoiceAmt,
                                df_InvoiceAddendum_Sub_Agg.SubscriptionStartDate,
                                df_InvoiceAddendum_Sub_Agg.PilotSubscriptionFlag
                                )
                                )
    # df_InvoiceAddendum_Sub_Agg.show()  #added for testing
    # df_CycleData_Subscription.show()   #added for testing
    df_CycleData_Subscription_SoldToCheck = (df_CycleData_Subscription.alias('df_CycleData_AGG')
                                        .join(df_InvoiceAddendum_Sub_Agg.alias("df_InvoiceAddendum"),
                                            (col("df_CycleData_AGG.CoolSculptingID") == col("df_InvoiceAddendum.CoolSculptingID")) &
                                            (col("df_CycleData_AGG.SoldToAccountID") == col("df_InvoiceAddendum.SoldToAccountID")) &
                                            (col("df_CycleData_AGG.ShipToAccountUUID") != col("df_InvoiceAddendum.ShipToAccountUUID")) &
                                            (col("df_CycleData_AGG.PromotionUUID") == col("df_InvoiceAddendum.PromotionUUID")) &
                                            (col("df_CycleData_AGG.CycleDate")
                                             .between(col("df_InvoiceAddendum.SubscriptionStartDate"),col("df_InvoiceAddendum.SubscriptionEndDate"))),
                                            'left'
                                            )
                                        .select(col("df_CycleData_AGG.CoolSculptingID"),
                                                col("df_CycleData_AGG.ShipToAccountID"),
                                                col("df_CycleData_AGG.ShipToAccountUUID"),
                                                col("df_CycleData_AGG.SoldToAccountID"),
                                                col("df_CycleData_AGG.PromotionUUID"),
                                                col("df_CycleData_AGG.CycleDate"),
                                                col("df_CycleData_AGG.CycleCount"),
                                                col("df_CycleData_AGG.NoPhoneNumberFlag"),
                                                col("df_CycleData_AGG.MaxInvoiceDate"),
                                                col("df_CycleData_AGG.MinCycleTmstmp"),
                                                col("df_CycleData_AGG.FraudFlag"),
                                                col("df_CycleData_AGG.MapFlag"),
                                                col("df_CycleData_AGG.TNCFlag"),
                                                col("df_CycleData_AGG.TNCPerCycleFlag"),
                                                col("df_CycleData_AGG.PerCycleFeeFlag"),
                                                col("df_CycleData_AGG.PerCycleFlag"),
                                                col("df_CycleData_AGG.OffBoarding_NonP3Flag"),
                                                col("df_CycleData_AGG.OffBoarding_NonP3_TreatmentFlag"),
                                                col("df_CycleData_AGG.OffBoarding_NonP3PerCycleFlag"),
                                                col("df_CycleData_AGG.OffBoarding_NonP3_TreatmentPerCycleFlag"),
                                                col("df_CycleData_AGG.EfficacyFlag"),
                                                col("df_CycleData_AGG.SmartCardSerialNumber"),
                                                col("df_CycleData_AGG.ApplicatorSerialNumber"),
                                                col("df_CycleData_AGG.CycleID"),
                                                col("df_CycleData_AGG.ConsumerSubscription_Comments"),
                                                coalesce(col("df_CycleData_AGG.ConsumerSubscriptionUUID"),col("df_InvoiceAddendum.ConsumerSubscriptionUUID")).alias('ConsumerSubscriptionUUID'),
                                                coalesce(col("df_CycleData_AGG.SubscriptionInvoiceAmt"),col("df_InvoiceAddendum.SubscriptionInvoiceAmt"),lit(0)).alias('SubscriptionInvoiceAmt'), 
                                                coalesce(col("df_CycleData_AGG.SubscriptionStartDate"),col("df_InvoiceAddendum.SubscriptionStartDate")).alias('SubscriptionStartDate'),
                                                coalesce(col("df_CycleData_AGG.PilotSubscriptionFlag"),col("df_InvoiceAddendum.PilotSubscriptionFlag")).alias('PilotSubscriptionFlag'),
                                                col("df_InvoiceAddendum.ShipToAccountUUID").alias("SubscriptionShipToAccountUUID")
                                                )
                                        .drop('InvoiceAddendumDetailsDeDupRowNum')
                                        .withColumn('ROW_NUM',row_number().over(Window.partitionBy('ConsumerSubscriptionUUID','PromotionUUID')
                                        .orderBy(asc('CycleDate'),asc('MinCycleTmstmp'),asc('ShipToAccountUUID'))))
                            )
    
    # df_CycleData_NoPhoneNumber = df_CycleData_AGG.filter('NoPhoneNumberFlag = 1')
    # df_CycleData_Sub_Classfication = df_CycleData_NoPhoneNumber.unionByName(df_CycleData_Subscription,allowMissingColumns=True)
    df_CycleData_Subscription_New = applyRules(df_CycleData_Subscription_SoldToCheck,'InvoiceAddendumDetails_Live')
    df_CycleData_Subscription_Classification = (df_CycleData_Subscription_New
                                                .join(df_PatientClassification, 
                                                        ['PatientClassificationName','PromotionUUID'],'left')
                                                .filter("CycleDate between PCL_EffectiveDate AND PCL_TerminationDate")
                                                .drop('PCL_TerminationDate','PCL_EffectiveDate')
                                                        )

    df_CycleData_Sub_Classfication_Agg = (
        df_CycleData_Subscription_Classification
        .groupBy(
            "ConsumerSubscriptionUUID",
            "CoolSculptingID",
            "ShipToAccountUUID",
            "SoldToAccountID",
            "PromotionUUID",
            "CycleDate",
            "PatientClassificationUUID",
            "Comments",
            "InvoiceCalculationType",
            "ListPrice",
            "MaxInvoiceDate"
        )
        .agg(
            sum("CycleCount").alias("CycleCount"),
            concat_ws(', ', collect_set('SmartCardSerialNumber')).alias('SmartCardSerialNumber'),
            concat_ws(', ', collect_set('ApplicatorSerialNumber')).alias('ApplicatorSerialNumber'),
            concat_ws(', ', collect_set('CycleID')).alias('CycleID')
        )
    )
    
    df_CycleData_Sub_Classfication_Final = applyRules(df_CycleData_Sub_Classfication_Agg,'InvoiceAddendumDetails_Live_InvoiceAmt')

    df_CycleData_InvoiceAddendum_Final_Merge = (df_CycleData_Sub_Classfication_Final
                                            .withColumn('NewInvoiceAddendumDetailsUUID',expr('uuid()'))
                                            .withColumn('VersionID',lit(1))
                                            .withColumn('CreatedBy',lit(UpdatedBy))
                                            .withColumn('CreatedDate',lit(current_timestamp()))
                                            .withColumn('UpdatedBy',lit(UpdatedBy))
                                            .withColumn('UpdatedDate',lit(current_timestamp()))
                                        )
    dl_FACTInvoiceAddendumDetails = DeltaTable.forName(spark, 'Promotion.FACT_InvoiceAddendumDetails')
    (dl_FACTInvoiceAddendumDetails.alias("tgt")
            .merge(df_CycleData_InvoiceAddendum_Final_Merge.alias("src"),
                ("src.CycleDate = tgt.CycleDate and tgt.ConsumerSubscriptionUUID = src.ConsumerSubscriptionUUID AND tgt.VersionID = src.VersionID AND tgt.PromotionUUID = src.PromotionUUID AND tgt.ShipToAccountUUID = src.ShipToAccountUUID"))
            .whenMatchedUpdate(set =
            {
                "tgt.CycleCount": "tgt.CycleCount+src.CycleCount",
                "tgt.Comments": "CASE WHEN tgt.CreatedDate <= src.MaxInvoiceDate AND tgt.CoolSculptingID <> 'MISSING' THEN 'UpdateCycle_NewSubscription' WHEN tgt.CreatedDate <= src.MaxInvoiceDate AND tgt.CoolSculptingID = 'MISSING' THEN 'UpdateCycle_NoPatientAssociation' ELSE tgt.Comments END",                
                "tgt.UpdatedBy": "src.UpdatedBy",
                "tgt.UpdatedDate": "src.UpdatedDate",
                "tgt.SmartCardSerialNumber": "concat_ws(', ', array_distinct(split(concat_ws(', ', tgt.SmartCardSerialNumber, src.SmartCardSerialNumber), ', ')))",
                "tgt.ApplicatorSerialNumber": "concat_ws(', ', array_distinct(split(concat_ws(', ', tgt.ApplicatorSerialNumber, src.ApplicatorSerialNumber), ', ')))",
                "tgt.CycleID": "concat_ws(', ', array_distinct(split(concat_ws(', ', tgt.CycleID, src.CycleID), ', ')))"
            })
            .whenNotMatchedInsert(values =
            {
                "tgt.InvoiceAddendumDetailsUUID": "src.NewInvoiceAddendumDetailsUUID",
                "tgt.ConsumerSubscriptionUUID": "src.ConsumerSubscriptionUUID",
                "tgt.CoolSculptingID": "src.CoolSculptingID",
                "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
                "tgt.SoldToAccountID": "src.SoldToAccountID",
                "tgt.PromotionUUID": "src.PromotionUUID",            
                "tgt.CycleDate": "src.CycleDate",
                "tgt.CycleCount": "src.CycleCount",    
                "tgt.IncrementalInvoiceCharge": "src.IncrementalInvoiceCharge",
                "tgt.PatientClassificationUUID": "src.PatientClassificationUUID",
                "tgt.VersionID": "src.VersionID",
                "tgt.Comments": "src.Comments",
                "tgt.CreatedBy": "src.CreatedBy",
                "tgt.CreatedDate": "src.CreatedDate",
                "tgt.UpdatedBy": "src.UpdatedBy",
                "tgt.UpdatedDate": "src.UpdatedDate",
                "tgt.SmartCardSerialNumber": "src.SmartCardSerialNumber",
                "tgt.ApplicatorSerialNumber": "src.ApplicatorSerialNumber",
                "tgt.CycleID": "src.CycleID"
            }
            )
            .execute())                              

In [0]:
def processInvoiceAddendumDetails_ConsumerBased(df_CycleData_AGG,UpdatedBy="ADB_UserSubscription"):
    
    df_InvoiceAddendumDetails = (spark.read.table('Promotion.FACT_InvoiceAddendumDetails')
                                            .filter('VersionId=1')
                                            .groupBy('ConsumerSubscriptionUUID')
                                            .sum('IncrementalInvoiceCharge')
                                            .withColumnRenamed('sum(IncrementalInvoiceCharge)','SubscriptionInvoiceAmt'))
    df_PatientClassification = (spark.read.table('Promotion.DIM_PatientClassification')
                                .select('PatientClassificationName','PromotionUUID', 'PatientClassificationUUID', 'ListPrice', 'InvoiceCalculationType', 
                        col('EffectiveDate').alias('PCL_EffectiveDate'),
                        col('TerminationDate').alias('PCL_TerminationDate'))
                                )
    df_ConsumerSubScription = spark.read.table('Promotion.FACT_ConsumerSubscription').filter('VersionId=1')
   
    df_InvoiceAddendum_Sub_Agg = (df_ConsumerSubScription
                                .join(df_InvoiceAddendumDetails,'ConsumerSubscriptionUUID','left')
                                .select(df_ConsumerSubScription.CoolSculptingID,
                                        df_ConsumerSubScription.PromotionUUID,
                                        df_ConsumerSubScription.SubscriptionStartDate,
                                        df_ConsumerSubScription.SubscriptionEndDate,
                                        df_ConsumerSubScription.ConsumerSubscriptionUUID,
                                        df_ConsumerSubScription.PilotSubscriptionFlag,
                                        df_ConsumerSubScription.Comments.alias('ConsumerSubscription_Comments'),
                                        df_InvoiceAddendumDetails.SubscriptionInvoiceAmt
                                        )
                                )

    df_CycleData_Subscription = (df_CycleData_AGG
                                #  .filter('NoPhoneNumberFlag = 0')
                        .join(df_InvoiceAddendum_Sub_Agg,
                                            (df_CycleData_AGG.CoolSculptingID == df_InvoiceAddendum_Sub_Agg.CoolSculptingID) &
                                            (df_CycleData_AGG.PromotionUUID == df_InvoiceAddendum_Sub_Agg.PromotionUUID) &
                                            (df_CycleData_AGG.CycleDate.between(df_InvoiceAddendum_Sub_Agg.SubscriptionStartDate,df_InvoiceAddendum_Sub_Agg.SubscriptionEndDate)) &
                                            ((df_CycleData_AGG.PerCycleFlag == 0) | (df_CycleData_AGG.PerCycleWhenNoSubscriptionFlag == 1)),
                                            'left'
                                            )
                        .select(df_CycleData_AGG.CoolSculptingID,
                                df_CycleData_AGG.ShipToAccountID,
                                df_CycleData_AGG.ShipToAccountUUID,
                                df_CycleData_AGG.SoldToAccountID,
                                df_CycleData_AGG.PromotionUUID,
                                df_CycleData_AGG.CycleDate,
                                df_CycleData_AGG.CycleCount,
                                df_CycleData_AGG.NoPhoneNumberFlag,
                                df_CycleData_AGG.MaxInvoiceDate,
                                df_CycleData_AGG.MinCycleTmstmp,
                                df_CycleData_AGG.FraudFlag,
                                df_CycleData_AGG.MapFlag,
                                df_CycleData_AGG.TNCFlag,     
                                df_CycleData_AGG.TNCPerCycleFlag,
                                df_CycleData_AGG.PerCycleFeeFlag,
                                df_CycleData_AGG.PerCycleFlag,
                                df_CycleData_AGG.OffBoarding_NonP3Flag,
                                df_CycleData_AGG.OffBoarding_NonP3_TreatmentFlag,
                                df_CycleData_AGG.OffBoarding_NonP3PerCycleFlag,
                                df_CycleData_AGG.OffBoarding_NonP3_TreatmentPerCycleFlag,
                                df_CycleData_AGG.EfficacyFlag,
                                df_CycleData_AGG.SmartCardSerialNumber,
                                df_CycleData_AGG.ApplicatorSerialNumber,
                                df_CycleData_AGG.CycleID,
                                df_InvoiceAddendum_Sub_Agg.ConsumerSubscription_Comments,
                                df_InvoiceAddendum_Sub_Agg.ConsumerSubscriptionUUID,
                                coalesce(df_InvoiceAddendum_Sub_Agg.SubscriptionInvoiceAmt, lit(0)).alias('SubscriptionInvoiceAmt'), 
                                df_InvoiceAddendum_Sub_Agg.SubscriptionStartDate,
                                df_InvoiceAddendum_Sub_Agg.PilotSubscriptionFlag
                                )
                        
                         .withColumn('ROW_NUM',row_number().over(Window.partitionBy('ConsumerSubscriptionUUID','PromotionUUID')
                                                        .orderBy(asc('CycleDate'),asc('MinCycleTmstmp'),asc('ShipToAccountUUID'))))
                        .withColumn('SubscriptionShipToAccountUUID', col('ShipToAccountUUID'))
                        .dropDuplicates()
                        ) 
    
    df_CycleData_Subscription_New = applyRules(df_CycleData_Subscription,'InvoiceAddendumDetails_Live')
    df_CycleData_Subscription_Classification = (df_CycleData_Subscription_New
                                                .join(df_PatientClassification, 
                                                        ['PatientClassificationName','PromotionUUID'],'left')
                                                .filter("CycleDate between PCL_EffectiveDate AND PCL_TerminationDate")
                                                .drop('PCL_TerminationDate','PCL_EffectiveDate')
                                                        )

    df_CycleData_Sub_Classfication_Agg = (
        df_CycleData_Subscription_Classification
        .groupBy(
            "ConsumerSubscriptionUUID",
            "CoolSculptingID",
            "ShipToAccountUUID",
            "SoldToAccountID",
            "PromotionUUID",
            "CycleDate",
            "PatientClassificationUUID",
            "Comments",
            "InvoiceCalculationType",
            "ListPrice",
            "MaxInvoiceDate"
        )
        .agg(
            sum("CycleCount").alias("CycleCount"),
            concat_ws(', ', collect_set('SmartCardSerialNumber')).alias('SmartCardSerialNumber'),
            concat_ws(', ', collect_set('ApplicatorSerialNumber')).alias('ApplicatorSerialNumber'),
            concat_ws(', ', collect_set('CycleID')).alias('CycleID')
        )
    )
    
    df_CycleData_Sub_Classfication_Final = applyRules(df_CycleData_Sub_Classfication_Agg,'InvoiceAddendumDetails_Live_InvoiceAmt')

    df_CycleData_InvoiceAddendum_Final_Merge = (df_CycleData_Sub_Classfication_Final
                                            .withColumn('NewInvoiceAddendumDetailsUUID',expr('uuid()'))
                                            .withColumn('VersionID',lit(1))
                                            .withColumn('CreatedBy',lit(UpdatedBy))
                                            .withColumn('CreatedDate',lit(current_timestamp()))
                                            .withColumn('UpdatedBy',lit(UpdatedBy))
                                            .withColumn('UpdatedDate',lit(current_timestamp()))
                                        )    

    dl_FACTInvoiceAddendumDetails = DeltaTable.forName(spark, 'Promotion.FACT_InvoiceAddendumDetails')
    (dl_FACTInvoiceAddendumDetails.alias("tgt")
            .merge(df_CycleData_InvoiceAddendum_Final_Merge.alias("src"),
                ("src.CycleDate = tgt.CycleDate and tgt.ConsumerSubscriptionUUID = src.ConsumerSubscriptionUUID AND tgt.VersionID = src.VersionID AND tgt.PromotionUUID = src.PromotionUUID AND tgt.ShipToAccountUUID = src.ShipToAccountUUID"))
            .whenMatchedUpdate(set =
            {
                "tgt.CycleCount": "tgt.CycleCount+src.CycleCount",
                "tgt.Comments": "CASE WHEN tgt.CreatedDate <= src.MaxInvoiceDate AND tgt.CoolSculptingID <> 'MISSING' THEN 'UpdateCycle_NewSubscription' WHEN tgt.CreatedDate <= src.MaxInvoiceDate AND tgt.CoolSculptingID = 'MISSING' THEN 'UpdateCycle_NoPatientAssociation' ELSE tgt.Comments END",                
                "tgt.UpdatedBy": "src.UpdatedBy",
                "tgt.UpdatedDate": "src.UpdatedDate",
                "tgt.SmartCardSerialNumber": "concat_ws(', ', array_distinct(split(concat_ws(', ', tgt.SmartCardSerialNumber, src.SmartCardSerialNumber), ', ')))",
                "tgt.ApplicatorSerialNumber": "concat_ws(', ', array_distinct(split(concat_ws(', ', tgt.ApplicatorSerialNumber, src.ApplicatorSerialNumber), ', ')))",
                "tgt.CycleID": "concat_ws(', ', array_distinct(split(concat_ws(', ', tgt.CycleID, src.CycleID), ', ')))"
            })
            .whenNotMatchedInsert(values =
            {
                "tgt.InvoiceAddendumDetailsUUID": "src.NewInvoiceAddendumDetailsUUID",
                "tgt.ConsumerSubscriptionUUID": "src.ConsumerSubscriptionUUID",
                "tgt.CoolSculptingID": "src.CoolSculptingID",
                "tgt.ShipToAccountUUID": "src.ShipToAccountUUID",
                "tgt.SoldToAccountID": "src.SoldToAccountID",
                "tgt.PromotionUUID": "src.PromotionUUID",            
                "tgt.CycleDate": "src.CycleDate",
                "tgt.CycleCount": "src.CycleCount",    
                "tgt.IncrementalInvoiceCharge": "src.IncrementalInvoiceCharge",
                "tgt.PatientClassificationUUID": "src.PatientClassificationUUID",
                "tgt.VersionID": "src.VersionID",
                "tgt.Comments": "src.Comments",
                "tgt.CreatedBy": "src.CreatedBy",
                "tgt.CreatedDate": "src.CreatedDate",
                "tgt.UpdatedBy": "src.UpdatedBy",
                "tgt.UpdatedDate": "src.UpdatedDate",
                "tgt.SmartCardSerialNumber": "src.SmartCardSerialNumber",
                "tgt.ApplicatorSerialNumber": "src.ApplicatorSerialNumber",
                "tgt.CycleID": "src.CycleID"
            }
            )
            .execute())                              